# Parse text for characters, creating a more useful form.

In [349]:
from opend6_tools.character import *
import re

In [350]:
text = """Walking Dead (Mummy, Skeleton, Zombie): Agility 2D, fighting 3D, Coordination 1D, Physiquev2D,vlifting 3D, Intellect 1D, Acumen
1D, search 3D, tracking 3D, Charisma 1D, intimidation 6D. Move: 10.
Strength Damage: 2D. Body Points: 15/Wound levels: 2. Disadvantages:
Employed (R3), slave to the one who raised them. Special Abilities:
Hardiness (R2), +2 to damage resistance totals; Immortality (R1),
cease functioning when smashed to pieces or head is cut off.
"""

In [351]:
name_detail = re.compile(r"(.*?):(.*)")
nd_split = name_detail.match(' '.join(text.splitlines()))
name = nd_split.group(1).strip()
detail = nd_split.group(2).strip()
detail

'Agility 2D, fighting 3D, Coordination 1D, Physiquev2D,vlifting 3D, Intellect 1D, Acumen 1D, search 3D, tracking 3D, Charisma 1D, intimidation 6D. Move: 10. Strength Damage: 2D. Body Points: 15/Wound levels: 2. Disadvantages: Employed (R3), slave to the one who raised them. Special Abilities: Hardiness (R2), +2 to damage resistance totals; Immortality (R1), cease functioning when smashed to pieces or head is cut off.'

In [352]:
def attr_skill_iter(detail: str):
    attr_skill = re.compile(r"([\w:\s]+)\s*(\d[D0](?:\+\d)?)[,\.]\s*")
    while m := attr_skill.match(detail):
        dice = m.group(2).replace("0", "D")
        yield m.group(1).strip(), f"{dice[:1]}*{dice[1:]}"
        detail = detail[m.end():]
        if detail.startswith("Move"):
            break
    yield detail

*attr_skills, detail = list(attr_skill_iter(detail))

In [353]:
attr_skills

[('Agility', '2*D'),
 ('fighting', '3*D'),
 ('Coordination', '1*D'),
 ('Physiquev', '2*D'),
 ('vlifting', '3*D'),
 ('Intellect', '1*D'),
 ('Acumen', '1*D'),
 ('search', '3*D'),
 ('tracking', '3*D'),
 ('Charisma', '1*D'),
 ('intimidation', '6*D')]

In [354]:
def attribute_arg_iter(attr_skills):
    attr_name = None
    skills = []
    for name, dice in attr_skills:
        if name in {"Agility", "Coordination", "Physique", "Intellect", "Acumen", "Charisma"}:
            if attr_name:
                if skills:
                    yield attr_name.lower(), f"{attr_name}({attr_dice}, {{{', '.join(skills)}}})"
                else:
                    yield attr_name.lower(), f"{attr_name}({attr_dice})"
            attr_name = name
            attr_dice = dice
            skills = []
        else:
            skills.append(f"{name!r}: {dice}")
    if skills:
        yield attr_name.lower(), f"{attr_name}({attr_dice}, {{{', '.join(skills)}}})"
    else:
        yield attr_name.lower(), f"{attr_name}({attr_dice})"

params = dict(attribute_arg_iter(attr_skills))
params

{'agility': "Agility(2*D, {'fighting': 3*D})",
 'coordination': "Coordination(1*D, {'Physiquev': 2*D, 'vlifting': 3*D})",
 'intellect': 'Intellect(1*D)',
 'acumen': "Acumen(1*D, {'search': 3*D, 'tracking': 3*D})",
 'charisma': "Charisma(1*D, {'intimidation': 6*D})"}

In [355]:
detail

'Move: 10. Strength Damage: 2D. Body Points: 15/Wound levels: 2. Disadvantages: Employed (R3), slave to the one who raised them. Special Abilities: Hardiness (R2), +2 to damage resistance totals; Immortality (R1), cease functioning when smashed to pieces or head is cut off.'

In [356]:
def other_features_iter(detail: str):
    feature = re.compile(r"([\w\s]+?):\s+([^\.]+?)\.\s*")
    while m := feature.match(detail):
        yield m.group(1).lower().replace(" ", "_"), repr(m.group(2))
        detail = detail[m.end():]

other = dict(other_features_iter(detail))
other

{'move': "'10'",
 'strength_damage': "'2D'",
 'body_points': "'15/Wound levels: 2'",
 'disadvantages': "'Employed (R3), slave to the one who raised them'",
 'special_abilities': "'Hardiness (R2), +2 to damage resistance totals; Immortality (R1), cease functioning when smashed to pieces or head is cut off'"}

In [357]:
print("Creature(")
print(f"    name={name!r},")
for name, value in (params | other).items():
    print(f"    {name}={value},")
print(")")

Creature(
    name='Walking Dead (Mummy, Skeleton, Zombie)',
    agility=Agility(2*D, {'fighting': 3*D}),
    coordination=Coordination(1*D, {'Physiquev': 2*D, 'vlifting': 3*D}),
    intellect=Intellect(1*D),
    acumen=Acumen(1*D, {'search': 3*D, 'tracking': 3*D}),
    charisma=Charisma(1*D, {'intimidation': 6*D}),
    move='10',
    strength_damage='2D',
    body_points='15/Wound levels: 2',
    disadvantages='Employed (R3), slave to the one who raised them',
    special_abilities='Hardiness (R2), +2 to damage resistance totals; Immortality (R1), cease functioning when smashed to pieces or head is cut off',
)
